# 02 - Feature engineering and target construction

Features are built in SQL (`sql/features/`). This notebook explains the design, checks the targets and looks
for leakage.

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
from src.db import read_sql
from src.evaluation import plots  # applies the shared chart style
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
FIG = Path("../reports/figures")

## 1. Two prediction problems, two time designs

| | PD model | Dormancy model |
|---|---|---|
| Unit | customer | customer x snapshot month |
| Features | Apr-Sep 2005 (6 months) | months t-1 and t |
| Target | missed payment in Oct 2005 | no balance and no new charges in t+1 **and** t+2 |
| Eligible | everyone | customers active in month t |
| Split | stratified customer split (one label month only) | train = May snapshot, test = Jul snapshot |

For the dormancy model the training labels (Jun-Jul) end before the test snapshot (end of Jul), so nothing from
the test period leaks into training. Sep 2005 is the scoring snapshot (predicts Oct-Nov, no labels yet).

In [2]:
read_sql('''SELECT snapshot_month_index AS snapshot, count(*) FILTER (WHERE is_eligible) eligible,
                  sum(churn_label) FILTER (WHERE is_eligible) dormant_next_2m,
                  avg(churn_label) FILTER (WHERE is_eligible) dormancy_rate
           FROM features.churn_snapshots GROUP BY 1 ORDER BY 1''')

,snapshot,eligible,dormant_next_2m,dormancy_rate
0,2,26057,426.0000,0.0163
1,3,26350,346.0000,0.0131
2,4,26681,323.0000,0.0121
3,5,27042,NaN,NaN
4,6,27606,NaN,NaN


Dormancy is rare (1.2-1.6% of active customers per snapshot), so accuracy would be meaningless:
a model predicting "nobody goes dormant" would be 98.5% accurate. I evaluate with PR-AUC, lift and capture
rate instead.

## 2. Leakage checks
1. The default label lives only in `core.fact_default_outcome`, and a test (`tests/test_features.py`) fails if any
   feature SQL file mentions it.
2. If a feature leaked the target it would separate the classes almost perfectly on its own. Single-feature
   ROC-AUCs are a quick check:

In [3]:
from sklearn.metrics import roc_auc_score
from src.modeling.features import PD_FEATURES, CHURN_FEATURES, load_pd_data, load_churn_data
pdd = load_pd_data()
rows = []
for f in [c for c in PD_FEATURES if c != "education"]:
    x = pdd[f].fillna(pdd[f].median())
    auc = roc_auc_score(pdd.target, x)
    rows.append((f, max(auc, 1 - auc)))
pd.DataFrame(rows, columns=["feature", "single_feature_auc"]).sort_values("single_feature_auc", ascending=False).head(10)

,feature,single_feature_auc
11,months_delinquent_6m,0.7261
13,months_since_delinquent,0.7185
12,months_60plus_6m,0.7142
10,max_status_6m,0.7077
14,delinquent_streak,0.6953
8,status_latest,0.6897
25,months_missed_payment,0.6470
23,payment_ratio_min,0.6461
9,status_prev,0.6386
5,new_charges_avg,0.6310


In [4]:
ch = load_churn_data(); ch = ch[ch.snapshot_month_index == 2]
rows = []
for f in [c for c in CHURN_FEATURES if c != "education"]:
    x = ch[f].fillna(ch[f].median())
    auc = roc_auc_score(ch.churn_label, x)
    rows.append((f, max(auc, 1 - auc)))
pd.DataFrame(rows, columns=["feature", "single_feature_auc"]).sort_values("single_feature_auc", ascending=False).head(10)

,feature,single_feature_auc
7,util_t,0.8425
8,util_tm1,0.8395
2,bill_tm1,0.8356
1,bill_t,0.8332
10,payment_to_limit_t,0.7254
12,payment_ratio_t,0.7180
3,payment_t,0.6990
11,charges_to_limit_t,0.6839
4,payment_tm1,0.6702
15,zero_balance_tm1,0.6561


The strongest single PD feature (months delinquent) reaches an AUC of about 0.73, and the strongest dormancy
feature (current utilization) about 0.84. That is strong but well short of the near-perfect separation a leaked
label would give, and it makes sense: a customer with almost no balance is the obvious dormancy candidate.

## 3. Do the main features relate to default in a sensible direction?

In [5]:
pdd["pay_ratio_bin"] = pd.qcut(pdd.payment_ratio_avg, 5, duplicates="drop")
pdd["util_bin"] = pd.cut(pdd.util_avg_6m, [-10, 0.1, 0.3, 0.5, 0.7, 0.9, 10])
display(pdd.groupby("months_since_delinquent").target.agg(customers="size", default_rate="mean"))
display(pdd.groupby("pay_ratio_bin", observed=True).target.agg(customers="size", default_rate="mean"))
display(pdd.groupby("util_bin", observed=True).target.agg(customers="size", default_rate="mean"))

,customers,default_rate
months_since_delinquent,,
0,6818,0.5029
1,447,0.3803
2,1175,0.2868
3,709,0.2581
4,406,0.2143
5,514,0.1868
6,19931,0.1171


,customers,default_rate
pay_ratio_bin,,
"(0.166, 0.203]",6000,0.2997
"(0.203, 0.234]",6000,0.2620
"(0.234, 0.533]",6000,0.1970
"(0.533, 1.0]",12000,0.1737


,customers,default_rate
util_bin,,
"(-10.0, 0.1]",11046,0.1881
"(0.1, 0.3]",4221,0.1547
"(0.3, 0.5]",3674,0.2066
"(0.5, 0.7]",3827,0.2537
"(0.7, 0.9]",4430,0.2847
"(0.9, 10.0]",2802,0.3262


Default risk falls the longer it has been since the last delinquency and the larger the share of the bill a
customer pays, and above ~10% utilization it rises steadily with utilization. The lowest-utilization group is
slightly riskier than the 10-30% group because it includes inactive accounts that the broad label still marks as
defaulting. These patterns match credit-risk intuition, which is a good sign the features were computed correctly
(independent recalculations are also in `tests/test_features.py`).

**Next:** modeling (`03_modeling.ipynb`).